# 🎙️ Narración automática: «Bukele en Haití»

**Un solo paso:** menú **Entorno de ejecución → Ejecutar todo** (o ⌘/Ctrl + F9).

- Si pregunta por la GPU: **Entorno de ejecución → Cambiar tipo → GPU T4 → Guardar** y vuelve a *Ejecutar todo*.
- Acepta el permiso de Google Drive cuando aparezca.
- Tarda unos 10–15 minutos. Al final se descarga **bukele_haiti_….wav**: arrástralo al chat con Claude.

Voz: *Narrador cercano (claro, joven)*, semilla 7, la que ya elegiste (se carga desde tu Drive).

In [ ]:
#@title ▶️ Instalar (ejecuta y espera al ✅)
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No hay GPU. Menú Entorno de ejecución → Cambiar tipo de entorno → GPU T4, y vuelve a ejecutar.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "num2words", "pyloudnorm", "jiwer"], check=True)
print("✅ Instalado. Si más abajo aparece un error de 'transformers', usa Entorno de ejecución → Reiniciar sesión y ejecuta desde la celda 2.")

In [ ]:
#@title ▶️ Conectar Drive (las voces y narraciones se guardan en MyDrive/Narrador)
usar_drive = True #@param {type:"boolean"}
from pathlib import Path
if usar_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/Narrador")
else:
    CARPETA = Path("/content/Narrador")
(CARPETA / "voces").mkdir(parents=True, exist_ok=True)
(CARPETA / "narraciones").mkdir(parents=True, exist_ok=True)
print("📁 Carpeta de trabajo:", CARPETA)

In [ ]:
%%writefile /content/texto.py
"""Preparación del texto en español: normalización para TTS y división en segmentos.

Qwen3-TTS lee mejor el texto "como se pronuncia": cifras, siglas y símbolos
escritos en palabras. Todo lo que no esté en el léxico se deja igual.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

try:
    from num2words import num2words
except ImportError:  # num2words es opcional; sin él las cifras se dejan como están
    num2words = None

LEXICO_POR_DEFECTO = Path("/content/lexico.json")


# ---------------------------------------------------------------- léxico
def cargar_lexico(ruta: str | Path | None = None) -> dict[str, str]:
    ruta = Path(ruta) if ruta else LEXICO_POR_DEFECTO
    if not ruta.exists():
        return {}
    datos = json.loads(ruta.read_text(encoding="utf-8"))
    return {k: v for k, v in datos.items() if not k.startswith("_")}


def aplicar_lexico(texto: str, lexico: dict[str, str]) -> str:
    # Las entradas más largas primero, para que "US$" no lo pise "US".
    for termino in sorted(lexico, key=len, reverse=True):
        patron = r"(?<![\wÁÉÍÓÚÑáéíóúñ])" + re.escape(termino) + r"(?![\wÁÉÍÓÚÑáéíóúñ])"
        texto = re.sub(patron, lexico[termino], texto)
    return texto


# ---------------------------------------------------------------- números
FEMENINOS = {
    "personas", "víctimas", "muertes", "escuelas", "tropas", "toneladas", "familias", "horas", "semanas",
    "mujeres", "niñas", "viviendas", "comunidades", "organizaciones", "elecciones", "vidas", "casas",
    "hectáreas", "ciudades", "deportaciones", "expulsiones", "camas", "plazas", "fuerzas", "bandas",
    "pandillas", "persona", "víctima", "escuela", "vez",
}


def _palabras(n: int | float, siguiente: str = "") -> str:
    if num2words is None:
        return str(n)
    t = num2words(n, lang="es")
    if siguiente.lower() in FEMENINOS:
        t = re.sub(r"ientos\b", "ientas", t)
        t = re.sub(r"\buno$", "una", t).replace("veintiuno", "veintiuna")
        t = re.sub(r"\buno\b(?= mil)", "un", t)
    elif siguiente and re.match(r"[a-záéíóúñ]", siguiente.lower()):
        t = re.sub(r"uno$", "ún" if t.endswith("veintiuno") else "un", t).replace("veintiún", "veintiún")
    return t


def _entero(cadena: str) -> int:
    return int(cadena.replace(".", "").replace("\u202f", "").replace(" ", ""))


_SIG = r"(?=(?:\s+(?:de\s+)?([a-záéíóúñ]+))?)"   # captura la palabra siguiente (para concordar género)


def normalizar_numeros(texto: str) -> str:
    if num2words is None:
        return texto
    # Rangos "80–90 %" -> "80 a 90 %"
    texto = re.sub(r"(\d+(?:,\d+)?)\s*[–-]\s*(\d+(?:,\d+)?)\s*%", r"\1 a \2 %", texto)
    # Porcentajes
    texto = re.sub(r"(\d+(?:,\d+)?)\s*%", r"\1 por ciento", texto)
    # Moneda: "US$ 880 millones" -> "880 millones de dólares"; "US$ 50" -> "50 dólares"
    texto = re.sub(r"US\$\s*(\d[\d.]*(?:,\d+)?)(\s+(?:mil\s+)?millones|\s+mil)?",
                   lambda m: f"{m.group(1)}{m.group(2) or ''} {'de ' if m.group(2) and 'millones' in m.group(2) else ''}dólares", texto)
    # Decimales con coma: "1,5" -> "uno coma cinco"
    def dec(m):
        ent, frac = m.group(1), m.group(2)
        frac_txt = _palabras(int(frac)) if not frac.startswith("0") else " ".join(_palabras(int(c)) for c in frac)
        return f"{_palabras(_entero(ent))} coma {frac_txt}"
    texto = re.sub(r"(?<![\d.])(\d{1,3}(?:\.\d{3})*|\d+),(\d+)(?![\d.]|,\d)", dec, texto)
    # Ordinales "1.º", "3.ª"
    texto = re.sub(r"(\d+)\.?\s*º(?=\s+[a-záéíóúñ])", lambda m: re.sub(r"(prim|terc)ero$", r"\1er", num2words(int(m.group(1)), lang="es", to="ordinal")), texto)
    texto = re.sub(r"(\d+)\.?\s*º", lambda m: num2words(int(m.group(1)), lang="es", to="ordinal"), texto)
    texto = re.sub(r"(\d+)\.?\s*ª", lambda m: num2words(int(m.group(1)), lang="es", to="ordinal")[:-1] + "a", texto)
    # Miles con punto o espacio fino: "4.217 muertes" -> "cuatro mil doscientas diecisiete muertes"
    texto = re.sub(r"(?<![\d,])(\d{1,3}(?:[.\u202f]\d{3})+)(?!\d|,\d)" + _SIG,
                   lambda m: _palabras(_entero(m.group(1)), m.group(2) or ""), texto)
    # Resto de enteros (años incluidos)
    texto = re.sub(r"(?<!\w)(?<!\d,)(\d+)(?!\w|,\d)" + _SIG,
                   lambda m: _palabras(int(m.group(1)), m.group(2) or ""), texto)
    return texto


def normalizar(texto: str, lexico: dict[str, str] | None = None) -> str:
    texto = texto.replace(" ", " ")
    texto = re.sub(r"\[\d+\]", "", texto)            # quita referencias tipo [12]
    texto = re.sub(r"\[[A-ZÁÉÍÓÚ ]+\]", "", texto)    # quita marcas tipo [VERIFICAR]
    # Rangos de años/cifras "1957–1986" -> "1957 a 1986" (los % se tratan después)
    texto = re.sub(r"(\d)\s*[–—-]\s*(\d[\d.,]*)(?!\s*%)", r"\1 a \2", texto)
    # Rayas de inciso -> pausa breve
    texto = re.sub(r"\s*[—–]\s*", ", ", texto)
    if lexico:
        texto = aplicar_lexico(texto, lexico)
    texto = normalizar_numeros(texto)
    texto = re.sub(r"\s+([.,;:!?…])", r"\1", texto)
    texto = re.sub(r",\s*,", ",", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    return texto.strip()


# ---------------------------------------------------------------- división
@dataclass
class Segmento:
    texto: str
    parrafo: int          # índice del párrafo (para pausas más largas entre párrafos)
    final_de_parrafo: bool


def dividir(texto: str, max_caracteres: int = 280) -> list[Segmento]:
    """Divide en párrafos (líneas en blanco) y, si un párrafo es largo,
    en grupos de frases completas de hasta `max_caracteres`.

    Segmentos cortos dan una voz más estable: el modelo pierde el hilo
    (repeticiones, cambios de ritmo) con entradas muy largas.
    """
    parrafos = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    segmentos: list[Segmento] = []
    for i, p in enumerate(parrafos):
        p = re.sub(r"\s*\n\s*", " ", p)
        frases = re.split(r"(?<=[.!?…;:])\s+(?=[¿¡\"“A-ZÁÉÍÓÚÑ0-9])", p)
        grupo = ""
        trozos: list[str] = []
        for f in frases:
            if len(f) > max_caracteres:  # frase larguísima: corta por comas
                for sub in re.split(r"(?<=,)\s+", f):
                    if grupo and len(grupo) + len(sub) + 1 > max_caracteres:
                        trozos.append(grupo); grupo = sub
                    else:
                        grupo = f"{grupo} {sub}".strip()
                continue
            if grupo and len(grupo) + len(f) + 1 > max_caracteres:
                trozos.append(grupo); grupo = f
            else:
                grupo = f"{grupo} {f}".strip()
        if grupo:
            trozos.append(grupo)
        for j, t in enumerate(trozos):
            segmentos.append(Segmento(texto=t, parrafo=i, final_de_parrafo=(j == len(trozos) - 1)))
    return segmentos

In [ ]:
#@title ▶️ Cargar motor y léxico de pronunciación
import json, math, re, time, gc
import numpy as np, soundfile as sf, torch
from IPython.display import Audio, display, HTML
import sys; sys.path.insert(0, "/content")
from texto import dividir, normalizar

# Léxico: a la izquierda cómo aparece en el guion, a la derecha cómo debe sonar. Edítalo aquí.
LEXICO = {
  "ONU": "Onu",
  "BINUH": "Binú",
  "OIM": "o i eme",
  "IPC": "i pe ce",
  "GSF": "ge ese efe",
  "CPT": "ce pe te",
  "CEP": "ce e pe",
  "OEA": "o e a",
  "OCHA": "Ocha",
  "UNICEF": "Unicef",
  "ACNUR": "Acnur",
  "CARICOM": "Caricom",
  "MSS": "eme ese ese",
  "UNSOH": "Unsó",
  "ACAPS": "Acaps",
  "ACLED": "Aclet",
  "MIREX": "Mírex",
  "EE. UU.": "Estados Unidos",
  "Croix-des-Bouquets": "Cruá de Buqué",
  "Viv Ansanm": "Viv Ansam",
  "Fils-Aimé": "Fis-Emé",
  "Jovenel Moïse": "Yovenel Moís",
  "Moïse": "Moís",
  "Artibonite": "Artibonít",
  "Duvalier": "Duvalié",
  "Haina": "Jaina",
  "Pedernales": "Pedernales",
  "Dajabón": "Dajabón",
  "kompa": "compá",
  "konbit": "conbít"
}
lex_dril = CARPETA / "lexico.json"
if lex_dril.exists():  # si ya guardaste un léxico en Drive, se usa ese
    LEXICO.update(json.loads(lex_dril.read_text(encoding="utf-8")))
lex_dril.write_text(json.dumps(LEXICO, ensure_ascii=False, indent=2), encoding="utf-8")

from qwen_tts import Qwen3TTSModel
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() and "T4" not in torch.cuda.get_device_name(0) else torch.float16
_MODELOS = {}
def modelo(tipo):
    nombre = {"diseno": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}[tipo]
    if nombre not in _MODELOS:
        print("Cargando", nombre, "… (la primera vez descarga ~4 GB)")
        _MODELOS[nombre] = Qwen3TTSModel.from_pretrained(nombre, device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
    return _MODELOS[nombre]

def liberar(tipo):
    nombre = {"diseno": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}[tipo]
    _MODELOS.pop(nombre, None); gc.collect(); torch.cuda.empty_cache()

def recortar(a, sr, umbral_db=-45, margen=0.04):
    v = int(0.01 * sr); n = len(a) // v
    if n == 0: return a
    rms = np.sqrt(np.mean(a[: n * v].reshape(n, v) ** 2, axis=1) + 1e-12)
    act = np.where(20 * np.log10(rms) > umbral_db)[0]
    if act.size == 0: return a
    return a[max(0, act[0] * v - int(margen * sr)): min(len(a), (act[-1] + 1) * v + int(margen * sr))]

def fundidos(a, sr, ms=12):
    k = min(len(a) // 2, int(ms / 1000 * sr))
    if k > 0:
        r = np.linspace(0, 1, k); a[:k] *= r; a[-k:] *= r[::-1]
    return a

def sonoridad(a, sr, lufs=-16.0):
    import pyloudnorm as pyln
    try:
        actual = pyln.Meter(sr).integrated_loudness(a)
        if math.isfinite(actual): a = pyln.normalize.loudness(a, actual, lufs)
    except Exception:
        a = a / (np.max(np.abs(a)) or 1) * 0.7
    p = np.max(np.abs(a))
    return a / p * 0.98 if p > 0.98 else a

def srt(manifiesto):
    def ts(x):
        h, r = divmod(x, 3600); m, s = divmod(r, 60)
        return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{int((s % 1) * 1000):03d}"
    return "\n".join(f"{m['n']}\n{ts(m['inicio'])} --> {ts(m['fin'])}\n{m['texto']}\n" for m in manifiesto)
print("✅ Motor listo.")

In [ ]:
#@title ▶️ Diseñar y escuchar la voz
voz = "Narrador cercano (claro, joven)" #@param ["Narrador documental (grave, pausado)", "Narrador cercano (claro, joven)", "Narradora documental (cálida, serena)", "Narradora periodística (clara, dinámica)", "Personalizada (escribe la descripción abajo)"]
descripcion_personalizada = "" #@param {type:"string"}
expresividad = "natural" #@param ["sobria", "natural", "expresiva"]
semilla = 7 #@param {type:"integer"}

VOCES = {
  "Narrador documental (grave, pausado)": "Adult male narrator in his mid-40s, deep warm baritone, native Latin American Spanish speaker with a neutral accent, crisp diction, calm and authoritative documentary delivery, measured pace.",
  "Narrador cercano (claro, joven)": "Male narrator around 30 years old, clear bright midrange voice, native neutral Latin American Spanish, friendly and engaging explainer tone, steady pace, very clear articulation.",
  "Narradora documental (cálida, serena)": "Adult female narrator in her early 40s, warm low-mezzo voice, native Latin American Spanish speaker with a neutral accent, serene and trustworthy documentary delivery, measured pace, crisp diction.",
  "Narradora periodística (clara, dinámica)": "Female news presenter around 35, clear confident voice, native neutral Latin American Spanish, dynamic yet serious journalistic delivery, precise articulation, moderate pace.",
}
ESTILO = {"sobria": (0.6, "Calm, measured, neutral documentary narration with minimal emotion."),
          "natural": (0.75, "Natural, warm documentary narration with gentle emphasis on key facts."),
          "expresiva": (0.9, "Engaging, expressive storytelling with clear emphasis on key figures and controlled emotion.")}
TEXTO_REF = ("Bienvenidos. En este documental recorreremos la historia de un país del Caribe, "
             "sus montañas, su gente y los desafíos que enfrenta hoy, con datos verificados y mucho respeto.")

desc = descripcion_personalizada.strip() if voz.startswith("Personalizada") else VOCES[voz]
if not desc:
    raise SystemExit("Escribe una descripción personalizada (en inglés funciona mejor).")
TEMP, estilo = ESTILO[expresividad]
import unicodedata
base_nombre = unicodedata.normalize("NFKD", voz if not voz.startswith("Personalizada") else desc[:30]).encode("ascii", "ignore").decode()
slug = re.sub(r"[^a-z0-9]+", "_", base_nombre.lower()).strip("_")
REF_WAV = CARPETA / "voces" / f"{slug}__{expresividad}__s{semilla}.wav"

if REF_WAV.exists():
    print("Ya tenías esta voz guardada; la reutilizo:", REF_WAV.name)
    ref_audio, SR = sf.read(REF_WAV, dtype="float32")
else:
    torch.manual_seed(semilla); np.random.seed(semilla)
    wavs, SR = modelo("diseno").generate_voice_design(text=TEXTO_REF, language="Spanish",
                                                       instruct=f"{desc} {estilo}", temperature=0.7)
    ref_audio = sonoridad(fundidos(recortar(np.asarray(wavs[0], dtype=np.float32), SR), SR), SR)
    sf.write(REF_WAV, ref_audio, SR, subtype="PCM_16")
    REF_WAV.with_suffix(".json").write_text(json.dumps({"voz": voz, "descripcion": desc, "expresividad": expresividad,
                                                        "semilla": semilla}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"🔊 {voz} · semilla {semilla} · {expresividad}")
display(Audio(ref_audio, rate=SR))
print("¿No te convence? Cambia la semilla (por ejemplo 8, 9, 10…) y vuelve a ejecutar. Apunta la que te guste.")

In [ ]:
#@title ✍️ Guion: Bukele en Haití (pega TODO esto en la celda del guion, reemplazando lo que haya)
LEXICO.update({"Sylvio Cator": "Silvio Catór", "Mawozo": "Mawoso", "Nayib": "Nayib"})

GUION = """
Cuatro soldados salvadoreños desaparecieron en Haití el 18 de octubre. Semanas después, más de 2.000 militares de El Salvador combaten a las pandillas en las calles de Puerto Príncipe. ¿Qué hace Nayib Bukele en Haití, y qué está en juego para los haitianos?

Para entenderlo hay que mirar primero a El Salvador. Desde marzo de 2022, el gobierno de Bukele mantiene un régimen de excepción que suspende garantías constitucionales y que ha llevado a prisión a más de 80.000 personas acusadas de pertenecer a pandillas. Los homicidios cayeron a mínimos históricos y Bukele se convirtió en uno de los líderes más populares de América Latina. Pero organizaciones de derechos humanos han documentado detenciones arbitrarias y cientos de muertes bajo custodia del Estado. Ese contraste, más seguridad a cambio de menos libertades, es lo que muchos llaman el modelo Bukele.

El Salvador no llegó a Haití de improviso. En 2024 ofreció ayuda contra las pandillas y envió un contingente reducido, que incluía personal de aviación, a la misión multinacional liderada por Kenia. Cuando esa misión se transformó en la Fuerza de Supresión de Pandillas de la ONU, la GSF, los salvadoreños siguieron allí. En septiembre de 2026, su Asamblea Legislativa extendió esa participación hasta octubre de 2027.

Todo cambió el 18 de octubre. En Croix-des-Bouquets, a las afueras de la capital, hombres armados atacaron un vehículo blindado de la GSF. Según el ejército salvadoreño, el vehículo fue alcanzado por disparos y explosivos y se incendió. Cuatro técnicos de aviación salvadoreños lograron salir, y desaparecieron. Ningún grupo ha reivindicado el ataque. A mediados de noviembre, su paradero sigue sin confirmarse oficialmente. Bukele incluso rechazó públicamente un reporte de prensa que aseguraba haber encontrado sus cuerpos.

La respuesta fue inmediata. Bukele anunció una operación de rescate, advirtió a las pandillas que sufrirían las consecuencias y ofreció retirar a todas sus fuerzas si los cuatro hombres eran liberados con vida. En pocas semanas, El Salvador desplegó entre 2.000 y 2.500 efectivos, con fuerzas especiales, helicópteros y vehículos blindados. El 12 de noviembre, tropas salvadoreñas junto a la policía y el ejército haitianos recuperaron el estadio nacional Sylvio Cator, que las bandas habían ocupado, y anunciaron que lo usarán como base para avanzar hacia barrios controlados por pandillas. El primer ministro haitiano, Alix Didier Fils-Aimé, agradeció el apoyo, y el presidente dominicano, Luis Abinader, respaldó el despliegue.

Pero la operación abre preguntas serias. La primera es legal. Según la ONU, solo unos 200 de estos soldados están bajo el mando de la GSF. El resto opera mediante un acuerdo bilateral con el gobierno haitiano que no se ha hecho público, y la propia GSF se ha desmarcado de sus acciones. Amnistía Internacional pidió publicar ese acuerdo y garantizar el respeto a los derechos humanos. La segunda pregunta es humana. En una operación contra una propiedad vinculada al líder de la banda 400 Mawozo, en Croix-des-Bouquets, murieron alrededor de 20 personas, según reportes de prensa. El observatorio ACLED contó al menos 21 muertes en operaciones de seguridad en esa zona entre el 18 de octubre y el 2 de noviembre, y Médicos Sin Fronteras atendió a unos 30 heridos por disparos y explosiones en la semana posterior a la emboscada. Una organización haitiana denunció bombardeos aéreos ante la ONU; Bukele lo niega. La tercera pregunta es diplomática. Estados Unidos, aliado cercano de Bukele, envió una nota formal de protesta asegurando que disparos salvadoreños alcanzaron su embajada en Puerto Príncipe, y criticó la falta de coordinación. El Salvador lo niega.

¿Puede funcionar el modelo Bukele en Haití? Sus defensores señalan resultados visibles en pocas semanas, algo que la misión internacional no había logrado. Sus críticos recuerdan que Haití no es El Salvador: no tiene un Estado con control del territorio, ni cárceles ni tribunales que funcionen con normalidad, ni autoridades electas. Y advierten que una fuerza extranjera que opera fuera del mando de la ONU, sin reglas públicas, puede dejar víctimas civiles sin rendición de cuentas. Hay también una lectura política: El Salvador celebrará elecciones presidenciales en 2027, y el despliegue en Haití proyecta la imagen de mano dura de Bukele más allá de sus fronteras.

Para millones de haitianos que viven bajo el control de las pandillas, cualquier fuerza que prometa recuperar las calles despierta esperanza. La pregunta es a qué costo, y quién responde por él. Lo que ocurra en las próximas semanas, con los cuatro técnicos desaparecidos y con los civiles atrapados en medio, dirá si Haití está ante una salida o ante un nuevo capítulo de intervención extranjera. Las fuentes están en la descripción.
"""


In [ ]:
#@title ▶️ Narrar y descargar
nombre_archivo = "bukele_haiti" #@param {type:"string"}
solo_vista_previa = False #@param {type:"boolean"}
pausa_entre_frases = 0.3 #@param {type:"number"}
pausa_entre_parrafos = 0.8 #@param {type:"number"}
ver_texto_leido = False #@param {type:"boolean"}

if "REF_WAV" not in globals():
    raise SystemExit("Primero ejecuta la celda 3 (elegir la voz).")
liberar("diseno")
base = modelo("base")
prompt = base.create_voice_clone_prompt(ref_audio=str(REF_WAV), ref_text=TEXTO_REF)

segmentos = dividir(GUION, 280)
if solo_vista_previa: segmentos = segmentos[:1]
if not segmentos: raise SystemExit("El guion está vacío.")

def generar(texto, s):
    torch.manual_seed(s); np.random.seed(s)
    w, sr = base.generate_voice_clone(text=texto, language="Spanish", voice_clone_prompt=prompt, temperature=TEMP)
    return recortar(np.asarray(w[0], dtype=np.float32), sr), sr

partes, manifiesto, t, inicio = [], [], 0.0, time.time()
for i, seg in enumerate(segmentos):
    leido = normalizar(seg.texto, LEXICO)
    if ver_texto_leido: print(f"   leído: {leido}")
    esperado = max(1.0, len(leido) / 14.5)
    mejor = None
    for intento in range(3):  # reintenta si la duración sale anómala (repeticiones o cortes)
        s = semilla + i + intento * 7919
        audio, SR = generar(leido, s)
        ratio = len(audio) / SR / esperado
        if mejor is None or abs(math.log(max(ratio, 1e-3))) < abs(math.log(max(mejor[1], 1e-3))):
            mejor = (audio, ratio, s)
        if 0.55 <= ratio <= 1.8: break
    audio, ratio, s = mejor
    audio = fundidos(audio, SR)
    dur = len(audio) / SR
    ok = 0.55 <= ratio <= 1.8
    print(f"[{i+1}/{len(segmentos)}] {'✅' if ok else '⚠️ revisar'} {dur:5.1f}s  {seg.texto[:70]}…")
    manifiesto.append({"n": i + 1, "texto": seg.texto, "leido": leido, "inicio": round(t, 3), "fin": round(t + dur, 3),
                       "semilla": s, "ratio": round(ratio, 2), "ok": ok})
    pausa = pausa_entre_parrafos if seg.final_de_parrafo else pausa_entre_frases
    partes += [audio, np.zeros(int(pausa * SR), np.float32)]
    t += dur + pausa

final = sonoridad(np.concatenate(partes[:-1]), SR)
destino = CARPETA / "narraciones" / f"{nombre_archivo}_{time.strftime('%Y%m%d_%H%M%S')}"
sf.write(destino.with_suffix(".wav"), final, SR, subtype="PCM_16")
destino.with_suffix(".srt").write_text(srt(manifiesto), encoding="utf-8")
destino.with_suffix(".json").write_text(json.dumps({"voz": REF_WAV.name, "segmentos": manifiesto}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ {len(final)/SR:.1f} s de audio en {time.time()-inicio:.0f} s → {destino.with_suffix('.wav')}")
display(Audio(final, rate=SR))
from google.colab import files
files.download(str(destino.with_suffix(".wav"))); files.download(str(destino.with_suffix(".srt")))